# Beam Tweaks: Retail Sales (March 2026)

Bob's direct-control workbench for micro-styling the five charts in the April 21, 2026 Beam. One cell per chart, all tweakable knobs (annotation coords, legend position, window, y-limits) at the top of each cell. Re-run a single cell to see the change inline, then run the final cell to rebuild the base64-inlined HTML preview at `/Users/bob/LHM/Outputs/beam_retail_sales_march2026/beam_preview.html`. Brand rules (palette, border, spines, watermarks) are locked at the top.

In [ ]:
# =============================================================================
# SETUP — brand constants, paths, helper functions. Don't tweak these.
# =============================================================================
import os
import sys
import base64
from datetime import datetime
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

# -----------------------------------------------------------------------------
# Paths — override the CSV paths if Bob is working on a different Beam run.
# -----------------------------------------------------------------------------
BASE = Path('/Users/bob/LHM/Outputs/beam_retail_sales_march2026')
OUT = BASE / 'charts'
OUT.mkdir(exist_ok=True)

CSV_FIG1 = BASE / 'fig1_headline_vs_control.csv'
CSV_FIG2 = BASE / 'fig2_food_services.csv'
CSV_FIG3 = BASE / 'fig3_march_by_category.csv'
CSV_FIG4 = BASE / 'fig4_real_pce_vs_retail_yoy.csv'
CSV_FIG5 = BASE / 'fig5_retail_vs_ip_yoy.csv'
CSV_RECESSIONS = BASE / 'recession_dates.csv'

HTML_PATH = BASE / 'beam_preview.html'

PULLED = 'April 21, 2026'

# -----------------------------------------------------------------------------
# Brand palette (23/89/BB mnemonic). Constants — do not edit.
# -----------------------------------------------------------------------------
COLORS = {
    'ocean':     '#2389BB',
    'dusk':      '#FF6723',
    'sky':       '#23BBFF',
    'venus':     '#FF2389',
    'sea':       '#00BB89',
    'doldrums':  '#898989',
    'starboard': '#238923',
    'port':      '#892323',
    'fog':       '#D1D1D1',
}

ICON_PATH = '/Users/bob/LHM/Brand/icon_transparent_128.png'
BG = '#ffffff'
FG = '#1a1a1a'
MUTED = '#555555'
SPINE = COLORS['doldrums']
DPI = 200
BORDER_WIDTH = 4.0

RIGHT_PAD_FRAC = 0.06  # uniform RHS pad across all time-series charts

# -----------------------------------------------------------------------------
# Helper functions — mirror the chart-god / lhm_chart_template style.
# -----------------------------------------------------------------------------
def new_fig(figsize=(14, 8)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    fig.subplots_adjust(top=0.88, bottom=0.08, left=0.06, right=0.94)
    return fig, ax

def style_ax(ax, right_primary=True):
    ax.grid(False)
    for s in ['top', 'right', 'left', 'bottom']:
        ax.spines[s].set_visible(True)
        ax.spines[s].set_linewidth(0.5)
        ax.spines[s].set_color(SPINE)
    ax.tick_params(colors=FG, labelsize=10)
    ax.xaxis.label.set_color(FG)
    ax.yaxis.label.set_color(FG)
    if right_primary:
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position('right')

def style_single_ax(ax, fmt='{:.1f}'):
    style_ax(ax, right_primary=True)
    ax.tick_params(axis='both', which='both', length=0)
    ax.tick_params(axis='y', labelcolor=FG, labelsize=10)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: fmt.format(x)))

def style_dual_ax(ax1, ax2, c_left, c_right):
    style_ax(ax1, right_primary=False)
    ax1.grid(False); ax2.grid(False)
    for spine in ax2.spines.values():
        spine.set_color(SPINE)
        spine.set_linewidth(0.5)
    ax1.tick_params(axis='both', which='both', length=0)
    ax1.tick_params(axis='y', labelcolor=c_left, labelsize=10)
    ax2.tick_params(axis='both', which='both', length=0)
    ax2.tick_params(axis='y', labelcolor=c_right, labelsize=10)
    ax1.yaxis.set_tick_params(which='both', right=False)
    ax2.yaxis.set_tick_params(which='both', left=False)

def align_yaxis_zero(a1, a2):
    y1_min, y1_max = a1.get_ylim()
    y2_min, y2_max = a2.get_ylim()
    r1 = abs(y1_min) / max(abs(y1_max), 1e-6)
    r2 = abs(y2_min) / max(abs(y2_max), 1e-6)
    r = max(r1, r2)
    a1.set_ylim(bottom=-r * abs(y1_max), top=y1_max)
    a2.set_ylim(bottom=-r * abs(y2_max), top=y2_max)

def set_xlim_right(ax, dates, pad_frac=RIGHT_PAD_FRAC, left=None, left_pad_days=0):
    """LHS flush by default (line-chart rule). RHS fractional pad for last-value pills."""
    data_min, data_max = dates.min(), dates.max()
    data_range = data_max - data_min
    right_pad = pd.Timedelta(seconds=data_range.total_seconds() * pad_frac)
    if left is None:
        left = data_min - pd.Timedelta(days=left_pad_days)
    ax.set_xlim(left, data_max + right_pad)

def add_annotation_pill(ax, text, x=0.50, y=0.95, ha='center'):
    """Signature contrast pill: white fill, Ocean border, Ocean italic text.
    x, y in axis coords (0-1). ha: 'left' | 'center' | 'right'."""
    ax.text(x, y, text, transform=ax.transAxes,
            fontsize=14, fontweight='bold', color=COLORS['ocean'],
            ha=ha, va='top', style='italic', zorder=20,
            bbox=dict(boxstyle='round,pad=0.5',
                      facecolor='#ffffff', edgecolor=COLORS['ocean'],
                      linewidth=1.5, alpha=1.0))

def add_last_value_pill(ax, value, color, side='right', fmt='{:+.2f}%', fontsize=9):
    """Colored pill at axis edge pinned at `value` (data y). side: 'right' | 'left'."""
    label = fmt.format(value)
    pill = dict(boxstyle='round,pad=0.3', facecolor=color, edgecolor=color, alpha=0.95)
    if side == 'right':
        ax.annotate(label, xy=(1.0, value), xycoords=('axes fraction', 'data'),
                    fontsize=fontsize, fontweight='bold', color='white',
                    ha='left', va='center', xytext=(6, 0),
                    textcoords='offset points', bbox=pill)
    else:
        ax.annotate(label, xy=(0.0, value), xycoords=('axes fraction', 'data'),
                    fontsize=fontsize, fontweight='bold', color='white',
                    ha='right', va='center', xytext=(-6, 0),
                    textcoords='offset points', bbox=pill)

def add_accent_bar(fig, y, height=0.004):
    bar = fig.add_axes([0.03, y, 0.94, height])
    bar.axhspan(0, 1, 0, 0.67, color=COLORS['ocean'])
    bar.axhspan(0, 1, 0.67, 1.0, color=COLORS['dusk'])
    bar.set_xlim(0, 1); bar.set_ylim(0, 1); bar.axis('off')

def add_watermarks(fig, title, subtitle, source, data_thru, pulled=PULLED):
    """Header band, accent bars, title/subtitle, source footer, MACRO ILLUMINATED."""
    fig.patch.set_facecolor(BG)
    fig.text(0.035, 0.98, 'LIGHTHOUSE MACRO', fontsize=13,
             color=COLORS['ocean'], fontweight='bold', va='top')
    fig.text(0.97, 0.98, datetime.now().strftime('%B %d, %Y'),
             fontsize=11, color=MUTED, ha='right', va='top')
    add_accent_bar(fig, 0.955)
    if os.path.exists(ICON_PATH):
        icon_img = mpimg.imread(ICON_PATH)
        icon_w = 0.018
        aspect = icon_img.shape[0] / icon_img.shape[1]
        fig_aspect = fig.get_figwidth() / fig.get_figheight()
        icon_h = icon_w * aspect * fig_aspect
        icon_ax = fig.add_axes([0.012, 0.985 - icon_h, icon_w, icon_h])
        icon_ax.imshow(icon_img, aspect='equal')
        icon_ax.axis('off')
    add_accent_bar(fig, 0.035)
    fig.text(0.97, 0.025, 'MACRO, ILLUMINATED.', fontsize=13,
             color=COLORS['ocean'], ha='right', va='top',
             style='italic', fontweight='bold')
    fig.text(0.03, 0.022,
             f'Lighthouse Macro | {source} | Data thru {data_thru} | Pulled {pulled}',
             fontsize=9, color=COLORS['doldrums'],
             ha='left', va='top', style='italic')
    fig.suptitle(title, fontsize=15, fontweight='bold', y=0.945, color=FG)
    if subtitle:
        fig.text(0.5, 0.895, subtitle, fontsize=14, ha='center',
                 color=COLORS['ocean'], style='italic')

def add_recession_shading(ax, start_date=None):
    rec = pd.read_csv(CSV_RECESSIONS, parse_dates=['start', 'end'])
    xmin = pd.Timestamp(start_date) if start_date is not None else None
    for _, r in rec.iterrows():
        if pd.isna(r['end']):
            continue
        if xmin is not None and r['end'] < xmin:
            continue
        s = max(r['start'], xmin) if xmin is not None else r['start']
        ax.axvspan(s, r['end'], color='gray', alpha=0.12, zorder=0)

def save_chart(fig, filename):
    """Save with 4pt Ocean border. Returns the path. Keeps figure open for inline display."""
    fig.patches.append(plt.Rectangle(
        (0, 0), 1, 1, transform=fig.transFigure,
        fill=False, edgecolor=COLORS['ocean'], linewidth=BORDER_WIDTH,
        zorder=100, clip_on=False,
    ))
    path = OUT / filename
    fig.savefig(path, dpi=DPI, bbox_inches='tight', pad_inches=0.025,
                facecolor=BG, edgecolor='none')
    return path

print('Setup loaded. Output dir:', OUT)


In [ ]:
# =============================================================================
# DATA FRESHNESS — last-modified + last observation per CSV.
# =============================================================================
for p in [CSV_FIG1, CSV_FIG2, CSV_FIG3, CSV_FIG4, CSV_FIG5]:
    mtime = datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
    try:
        df = pd.read_csv(p)
        if 'date' in df.columns:
            last_obs = pd.to_datetime(df['date']).max().strftime('%Y-%m-%d')
        elif 'reference_month' in df.columns:
            last_obs = pd.to_datetime(df['reference_month']).max().strftime('%Y-%m-%d')
        else:
            last_obs = f'{len(df)} rows'
    except Exception as e:
        last_obs = f'read error: {e}'
    print(f'{p.name:45s}  mod {mtime}  last {last_obs}')


## Fig 1 — Headline vs control group (24m, dual axis)

In [ ]:
# --- TWEAKABLE --------------------------------------------------------------
F1_ANNOTATION = 'March: headline +1.66%, control +0.76% vs +0.2% consensus.'
F1_ANNOTATION_X = 0.98
F1_ANNOTATION_Y = 0.14
F1_ANNOTATION_HA = 'right'
F1_TITLE = 'March headline surged. The control group kept pace, not caught fire.'
F1_SUBTITLE = 'Figure 1 | Retail sales m/m: headline vs control group (24m)'
F1_SOURCE = 'U.S. Census Bureau'
F1_DATA_THRU = 'March 2026'
# ----------------------------------------------------------------------------

df = pd.read_csv(CSV_FIG1, parse_dates=['date'])

fig, ax1 = new_fig(figsize=(14, 8))
ax2 = ax1.twinx()

ax1.plot(df['date'], df['retail_headline_mom_pct'],
         color=COLORS['ocean'], linewidth=3.0, label='Headline retail')
ax2.plot(df['date'], df['retail_control_group_mom_pct'],
         color=COLORS['dusk'], linewidth=3.0, label='Control group')

style_dual_ax(ax1, ax2, c_left=COLORS['dusk'], c_right=COLORS['ocean'])
ax1.axhline(0, color=COLORS['fog'], linestyle='--', linewidth=0.8, zorder=0)

# RHS primary conventions
ax1.yaxis.tick_right(); ax1.yaxis.set_label_position('right')
ax2.yaxis.tick_left(); ax2.yaxis.set_label_position('left')
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x:+.1f}%'))
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x:+.1f}%'))
align_yaxis_zero(ax1, ax2)

add_last_value_pill(ax1, float(df['retail_headline_mom_pct'].iloc[-1]),
                    COLORS['ocean'], side='right', fmt='+{:.2f}%')
add_last_value_pill(ax2, float(df['retail_control_group_mom_pct'].iloc[-1]),
                    COLORS['dusk'], side='left', fmt='+{:.2f}%')

set_xlim_right(ax1, df['date'])
add_annotation_pill(ax1, F1_ANNOTATION, x=F1_ANNOTATION_X, y=F1_ANNOTATION_Y, ha=F1_ANNOTATION_HA)

add_watermarks(fig, title=F1_TITLE, subtitle=F1_SUBTITLE, source=F1_SOURCE, data_thru=F1_DATA_THRU)
path = save_chart(fig, 'fig1.png')
plt.show()
print('wrote', path)


## Fig 2 — Food services & drinking places (24m, bars)

In [ ]:
# --- TWEAKABLE --------------------------------------------------------------
F2_ANNOTATION = 'Feb +0.42%. Mar +0.14%. Weakest in six months.'
F2_ANNOTATION_X = 0.98
F2_ANNOTATION_Y = 0.95
F2_ANNOTATION_HA = 'right'
F2_LEFT_PAD_DAYS = 30       # bar-chart LHS breathing room
F2_BAR_WIDTH = 22
F2_TITLE = 'The one services line decelerated while headline fireworks went off.'
F2_SUBTITLE = 'Figure 2 | Food services & drinking places, m/m (24m)'
F2_SOURCE = 'U.S. Census Bureau'
F2_DATA_THRU = 'March 2026'
# ----------------------------------------------------------------------------

df = pd.read_csv(CSV_FIG2, parse_dates=['date'])

fig, ax = new_fig(figsize=(14, 8))

bar_colors = [COLORS['ocean'] if v >= 0 else COLORS['port']
              for v in df['food_services_mom_pct']]
ax.bar(df['date'], df['food_services_mom_pct'],
       width=F2_BAR_WIDTH, color=bar_colors, edgecolor='none', zorder=3)

style_single_ax(ax, fmt='{:+.1f}%')
ax.axhline(0, color=COLORS['fog'], linestyle='--', linewidth=0.8, zorder=0)

# Last-value pill anchored at the last bar top (not the axis edge)
last_date = df['date'].iloc[-1]
last_val = float(df['food_services_mom_pct'].iloc[-1])
ax.annotate(f'+{last_val:.2f}%',
            xy=(last_date, last_val),
            xytext=(12, 0), textcoords='offset points',
            fontsize=10, fontweight='bold', color='white',
            ha='left', va='center',
            bbox=dict(boxstyle='round,pad=0.35',
                      facecolor=COLORS['ocean'],
                      edgecolor=COLORS['ocean'], alpha=0.95))

set_xlim_right(ax, df['date'], left_pad_days=F2_LEFT_PAD_DAYS)
add_annotation_pill(ax, F2_ANNOTATION, x=F2_ANNOTATION_X, y=F2_ANNOTATION_Y, ha=F2_ANNOTATION_HA)

add_watermarks(fig, title=F2_TITLE, subtitle=F2_SUBTITLE, source=F2_SOURCE, data_thru=F2_DATA_THRU)
path = save_chart(fig, 'fig2.png')
plt.show()
print('wrote', path)


## Fig 3 — March m/m by category (13 horizontal bars)

In [ ]:
# --- TWEAKABLE --------------------------------------------------------------
F3_ANNOTATION = 'Gasoline led on price, not volume (+15.45%).\nFood services just +0.14%.'
F3_ANNOTATION_X = 0.98
F3_ANNOTATION_Y = 0.30
F3_ANNOTATION_HA = 'right'
F3_XLIM_LEFT = -1.1            # tight to the -0.91% bar tip
F3_XLIM_RIGHT_PAD = 1.5        # extra headroom past the gasoline bar
F3_LEFT_MARGIN = 0.20          # subplot left margin (clears long category labels)
F3_LEGEND_LOC = 'lower right'
F3_TITLE = 'Twelve of thirteen categories green. Gasoline did most of the work.'
F3_SUBTITLE = 'Figure 3 | March 2026 retail sales by category, m/m'
F3_SOURCE = 'U.S. Census Bureau'
F3_DATA_THRU = 'March 2026'
# ----------------------------------------------------------------------------

df = pd.read_csv(CSV_FIG3)
df = df.sort_values('mom_pct', ascending=True).reset_index(drop=True)

def bar_color(cat, val):
    if cat == 'Gasoline Stations':
        return COLORS['dusk']
    if cat == 'Food Services & Drinking Places':
        return COLORS['sea']
    if cat == 'Miscellaneous Store Retailers' and val < 0:
        return COLORS['port']
    return COLORS['ocean']

colors = [bar_color(c, v) for c, v in zip(df['category'], df['mom_pct'])]

fig, ax = new_fig(figsize=(14, 9))
fig.subplots_adjust(left=F3_LEFT_MARGIN, right=0.94, top=0.88, bottom=0.08)
bars = ax.barh(df['category'], df['mom_pct'],
               color=colors, edgecolor='none', zorder=3)

ax.grid(False)
for s in ['top', 'right', 'left', 'bottom']:
    ax.spines[s].set_visible(True)
    ax.spines[s].set_linewidth(0.5)
    ax.spines[s].set_color(SPINE)
ax.tick_params(axis='both', which='both', length=0, labelsize=10)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x:+.0f}%'))
ax.axvline(0, color=COLORS['fog'], linestyle='--', linewidth=0.8, zorder=0)

for bar, val, color in zip(bars, df['mom_pct'], colors):
    y = bar.get_y() + bar.get_height() / 2
    if val >= 0:
        ax.annotate(f'+{val:.2f}%', xy=(val, y),
                    xytext=(6, 0), textcoords='offset points',
                    fontsize=9, fontweight='bold', color='white',
                    ha='left', va='center',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor=color, edgecolor=color, alpha=0.95),
                    zorder=5)
    else:
        ax.annotate(f'{val:.2f}%', xy=(0, y),
                    xytext=(6, 0), textcoords='offset points',
                    fontsize=9, fontweight='bold', color='white',
                    ha='left', va='center',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor=color, edgecolor=color, alpha=0.95),
                    zorder=5)

x_max = df['mom_pct'].max()
ax.set_xlim(F3_XLIM_LEFT, x_max + x_max * 0.18 + F3_XLIM_RIGHT_PAD)

legend_handles = [
    mpatches.Patch(color=COLORS['dusk'], label='Price-driven'),
    mpatches.Patch(color=COLORS['sea'], label='Services'),
    mpatches.Patch(color=COLORS['port'], label='Declined'),
]
leg = ax.legend(handles=legend_handles, loc=F3_LEGEND_LOC, fontsize=10,
                frameon=True, framealpha=0.95, facecolor='#ffffff',
                edgecolor=COLORS['doldrums'],
                borderpad=0.6, handlelength=1.2, handleheight=1.0)
leg.set_zorder(15)

add_annotation_pill(ax, F3_ANNOTATION, x=F3_ANNOTATION_X, y=F3_ANNOTATION_Y, ha=F3_ANNOTATION_HA)

add_watermarks(fig, title=F3_TITLE, subtitle=F3_SUBTITLE, source=F3_SOURCE, data_thru=F3_DATA_THRU)
path = save_chart(fig, 'fig3.png')
plt.show()
print('wrote', path)


## Fig 4 — Real PCE YoY vs headline retail YoY (post-pandemic)

In [ ]:
# --- TWEAKABLE --------------------------------------------------------------
F4_WINDOW_START = '2021-07-01'   # trims post-COVID base effect. Override to None for full history.
F4_ANNOTATION = 'Real PCE decelerating. Nominal retail near 4% YoY. Price, not volume.'
F4_ANNOTATION_X = 0.95
F4_ANNOTATION_Y = 0.80
F4_ANNOTATION_HA = 'right'
F4_LEGEND_LOC = 'upper right'
F4_YLIM = None                   # e.g. (-2, 20) to override auto
F4_TITLE = 'The real economy is softening. The nominal headline does not know it.'
F4_SUBTITLE = 'Figure 4 | Real PCE YoY vs nominal headline retail YoY (post-pandemic)'
F4_SOURCE = 'Bureau of Economic Analysis, U.S. Census Bureau'
F4_DATA_THRU = 'February 2026'
# ----------------------------------------------------------------------------

df = pd.read_csv(CSV_FIG4, parse_dates=['date'])
if F4_WINDOW_START is not None:
    df = df[df['date'] >= F4_WINDOW_START].reset_index(drop=True)

fig, ax = new_fig(figsize=(14, 8))

ax.plot(df['date'], df['real_pce_yoy_pct'],
        color=COLORS['ocean'], linewidth=3.0, label='Real PCE YoY')
ax.plot(df['date'], df['retail_headline_yoy_pct'],
        color=COLORS['dusk'], linewidth=3.0, label='Nominal retail YoY')

style_single_ax(ax, fmt='{:+.0f}%')
ax.axhline(0, color=COLORS['fog'], linestyle='--', linewidth=0.8, zorder=0)

if F4_YLIM is None:
    y_all = pd.concat([df['real_pce_yoy_pct'], df['retail_headline_yoy_pct']])
    y_max, y_min = float(y_all.max()), float(y_all.min())
    pad = (y_max - y_min) * 0.06
    ax.set_ylim(y_min - pad, y_max + pad)
else:
    ax.set_ylim(*F4_YLIM)

set_xlim_right(ax, df['date'])

# Stacked last-value pills on RHS (retail higher, PCE lower)
last_retail = float(df['retail_headline_yoy_pct'].iloc[-1])
last_pce = float(df['real_pce_yoy_pct'].iloc[-1])
add_last_value_pill(ax, last_retail, COLORS['dusk'], side='right', fmt='+{:.1f}%')
add_last_value_pill(ax, last_pce, COLORS['ocean'], side='right', fmt='+{:.1f}%')

legend_handles = [
    mpatches.Patch(color=COLORS['ocean'], label='Real PCE YoY'),
    mpatches.Patch(color=COLORS['dusk'], label='Nominal retail YoY'),
]
leg = ax.legend(handles=legend_handles, loc=F4_LEGEND_LOC, fontsize=10,
                frameon=True, framealpha=0.95, facecolor='#ffffff',
                edgecolor=COLORS['doldrums'],
                borderpad=0.6, handlelength=1.2, handleheight=1.0)
leg.set_zorder(15)

add_annotation_pill(ax, F4_ANNOTATION, x=F4_ANNOTATION_X, y=F4_ANNOTATION_Y, ha=F4_ANNOTATION_HA)

add_watermarks(fig, title=F4_TITLE, subtitle=F4_SUBTITLE, source=F4_SOURCE, data_thru=F4_DATA_THRU)
path = save_chart(fig, 'fig4.png')
plt.show()
print('wrote', path)


## Fig 5 — Retail vs Industrial Production YoY (33Y, NBER shading)

In [ ]:
# --- TWEAKABLE --------------------------------------------------------------
F5_WINDOW_START = None           # e.g. '1990-01-01' to trim; None = full CSV
F5_ANNOTATION = 'March: retail +4.0% YoY.\nIP +0.7% YoY, but -0.5% m/m.\nDemand up, supply turning.'
F5_ANNOTATION_X = 0.10
F5_ANNOTATION_Y = 0.60
F5_ANNOTATION_HA = 'left'
F5_LEGEND_LOC = 'upper right'
F5_YLIM = None                   # e.g. (-25, 55) to override auto
F5_TITLE = 'Demand and supply are telling different stories.'
F5_SUBTITLE = 'Figure 5 | Retail sales vs Industrial Production, YoY (33Y)'
F5_SOURCE = 'U.S. Census Bureau, Federal Reserve'
F5_DATA_THRU = 'March 2026'
# ----------------------------------------------------------------------------

df = pd.read_csv(CSV_FIG5, parse_dates=['date'])
if F5_WINDOW_START is not None:
    df = df[df['date'] >= F5_WINDOW_START].reset_index(drop=True)

fig, ax = new_fig(figsize=(14, 8))

ax.plot(df['date'], df['retail_headline_yoy_pct'],
        color=COLORS['ocean'], linewidth=3.0, label='Retail sales YoY')
ax.plot(df['date'], df['industrial_production_yoy_pct'],
        color=COLORS['dusk'], linewidth=3.0, label='Industrial Production YoY')

style_single_ax(ax, fmt='{:+.0f}%')
ax.axhline(0, color=COLORS['fog'], linestyle='--', linewidth=0.8, zorder=0)

if F5_YLIM is None:
    y_all = pd.concat([df['retail_headline_yoy_pct'],
                       df['industrial_production_yoy_pct']])
    y_max, y_min = float(y_all.max()), float(y_all.min())
    pad = (y_max - y_min) * 0.05
    ax.set_ylim(y_min - pad, y_max + pad)
else:
    ax.set_ylim(*F5_YLIM)

add_recession_shading(ax, start_date=df['date'].min())
set_xlim_right(ax, df['date'])

last_retail = float(df['retail_headline_yoy_pct'].iloc[-1])
last_ip = float(df['industrial_production_yoy_pct'].iloc[-1])
add_last_value_pill(ax, last_retail, COLORS['ocean'], side='right', fmt='+{:.1f}%')
add_last_value_pill(ax, last_ip, COLORS['dusk'], side='right', fmt='+{:.1f}%')

legend_handles = [
    mpatches.Patch(color=COLORS['ocean'], label='Retail sales YoY'),
    mpatches.Patch(color=COLORS['dusk'], label='Industrial Production YoY'),
]
leg = ax.legend(handles=legend_handles, loc=F5_LEGEND_LOC, fontsize=10,
                frameon=True, framealpha=0.95, facecolor='#ffffff',
                edgecolor=COLORS['doldrums'],
                borderpad=0.6, handlelength=1.2, handleheight=1.0)
leg.set_zorder(15)

add_annotation_pill(ax, F5_ANNOTATION, x=F5_ANNOTATION_X, y=F5_ANNOTATION_Y, ha=F5_ANNOTATION_HA)

add_watermarks(fig, title=F5_TITLE, subtitle=F5_SUBTITLE, source=F5_SOURCE, data_thru=F5_DATA_THRU)
path = save_chart(fig, 'fig5.png')
plt.show()
print('wrote', path)


## Rebuild HTML preview (base64-inlined, self-contained)

In [ ]:
# =============================================================================
# Rebuild beam_preview.html with the freshly saved PNGs inlined as base64.
# Body copy is held here as constants so the cell is self-contained. If Bob
# edits prose, edit here and re-run.
# =============================================================================
BEAM_TITLE = 'A 1.7% Print, a 0.14% Tell'
BEAM_DATE_STR = 'April 21, 2026'
BEAM_BYLINE = f'Lighthouse Macro | The Beam | {BEAM_DATE_STR}'

def b64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('ascii')

IMG1 = b64(OUT / 'fig1.png')
IMG2 = b64(OUT / 'fig2.png')
IMG3 = b64(OUT / 'fig3.png')
IMG4 = b64(OUT / 'fig4.png')
IMG5 = b64(OUT / 'fig5.png')

def img(b):
    return f'data:image/png;base64,{b}'

html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{BEAM_TITLE} \u2014 Lighthouse Macro | The Beam</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=Montserrat:wght@600;700;800&display=swap" rel="stylesheet">
<style>
  :root {{
    --ocean: #2389BB;
    --dusk: #FF6723;
    --doldrums: #898989;
    --ink: #1a1a1a;
    --fog: #D1D1D1;
  }}
  * {{ box-sizing: border-box; }}
  html, body {{
    margin: 0; padding: 0; background: #ffffff; color: var(--ink);
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
    font-size: 16px; line-height: 1.6;
    -webkit-font-smoothing: antialiased;
    -moz-osx-font-smoothing: grayscale;
  }}
  .wrap {{ max-width: 780px; margin: 0 auto; padding: 56px 24px 80px 24px; }}
  h1.title {{
    font-family: 'Montserrat', sans-serif; font-weight: 800;
    font-size: 32pt; line-height: 1.15; color: var(--ink);
    margin: 0 0 14px 0; letter-spacing: -0.01em;
  }}
  .accent-bar {{ display: flex; width: 100%; height: 5px; margin: 6px 0 18px 0; }}
  .accent-bar .a {{ width: 66.6667%; background: var(--ocean); }}
  .accent-bar .b {{ width: 33.3333%; background: var(--dusk); }}
  .byline {{
    font-family: 'Inter', sans-serif; font-size: 13px; color: var(--doldrums);
    letter-spacing: 0.04em; text-transform: none; margin: 0 0 40px 0;
  }}
  h2.section {{
    font-family: 'Montserrat', sans-serif; font-weight: 700;
    font-size: 22pt; line-height: 1.2; color: var(--ink);
    margin: 44px 0 18px 0; letter-spacing: -0.005em;
  }}
  p {{ margin: 0 0 18px 0; font-size: 17px; line-height: 1.65; }}
  figure {{ margin: 28px 0 28px 0; }}
  figure img {{ display: block; width: 100%; max-width: 100%; height: auto; border-radius: 2px; }}
  figcaption {{
    font-family: 'Inter', sans-serif; font-style: italic; font-size: 13pt;
    color: var(--doldrums); text-align: center; margin-top: 10px;
  }}
  .footer {{
    margin-top: 64px; padding-top: 24px; border-top: 1px solid var(--fog);
    font-size: 13px; color: var(--doldrums); text-align: center; letter-spacing: 0.02em;
  }}
</style>
</head>
<body>
<div class="wrap">

  <h1 class="title">{BEAM_TITLE}</h1>
  <div class="accent-bar"><div class="a"></div><div class="b"></div></div>
  <div class="byline">{BEAM_BYLINE}</div>

  <h2 class="section">Retail sales surged. The service line didn\u2019t.</h2>

  <p>March retail sales rose 1.66% month over month, the biggest monthly gain in more than a year, and twelve of the report\u2019s thirteen categories were green. The control group, the measure that feeds directly into GDP, climbed 0.76% against a 0.2% consensus. Bloomberg ran it as broad-based resilience. Benzinga ran it as consumers shrugging off the pump shock. The tape bought the number early, then gave most of it back into the close.</p>

  <p>Headlines did their job. The composition is where the story actually lives.</p>

  <figure>
    <img src="{img(IMG1)}" alt="Figure 1">
    <figcaption>Figure 1: March retail sales, headline vs control group (m/m)</figcaption>
  </figure>

  <h2 class="section">What the numbers actually say</h2>

  <p>Gasoline stations rose 15.45% on the month, almost entirely on price. Middle East tension pushed pump prices higher through the back half of the month, and headline retail sales are nominal, so every tick at the pump shows up as sales growth. Behind gasoline, the biggest monthly gainers were furniture (+2.2%), nonstore retailers (+1.0%), and general merchandise (+1.0%). The NRF also noted refund season ran hotter than usual this cycle, front-loading March spending.</p>

  <p>The one services line in the report, food services and drinking places, rose 0.14%. That is a deceleration from February and the weakest print in six months. Clothing printed +0.03%. Sporting goods printed +0.04%. Miscellaneous retailers were the one red category, down 0.91%. Year over year, headline retail sales held near 4%, essentially unchanged from February. The monthly fireworks didn\u2019t move the annual trajectory at all.</p>

  <p>Industrial production was up 0.7% year over year in March. But it fell 0.5% month over month. Demand up, supply down on the turn. That gap has to close somewhere.</p>

  <figure>
    <img src="{img(IMG2)}" alt="Figure 2">
    <figcaption>Figure 2: Food services &amp; drinking places (m/m)</figcaption>
  </figure>

  <h2 class="section">Why it matters: the CCI decomposition</h2>

  <figure>
    <img src="{img(IMG3)}" alt="Figure 3">
    <figcaption>Figure 3: March retail sales by category (m/m)</figcaption>
  </figure>

  <p>This is where the Consumer Conditions Index earns its keep. CCI is Pillar 5 in our framework, the last domino in the labor-to-credit-to-equity chain, and it\u2019s weighted deliberately toward real measures rather than nominal headlines. Real PCE YoY gets 25% of the index. The retail sales control group gets 15%. Nominal headline retail sales get zero. The index was built this way because the composition traps in monthly retail data are exactly what we\u2019re looking at right now.</p>

  <figure>
    <img src="{img(IMG4)}" alt="Figure 4">
    <figcaption>Figure 4: Real PCE vs nominal retail sales (YoY)</figcaption>
  </figure>

  <h2 class="section">What consensus is missing</h2>

  <p>The print is nominal. Strip gasoline stations alone and the headline falls from 1.66% to roughly 0.6% on the month. Consumer strength that requires a 15% pump-price spike to clear the 1% line is a different animal from consumer strength powered by rising real wages and sustained volume growth.</p>

  <p>The industrial production divergence is the other tell. Retail sales up 1.66% with IP down 0.5% in the same month is not a signal of healthy demand meeting healthy supply. It\u2019s a signal that margin pressure is accumulating somewhere, most likely inside Q1 corporate earnings, and that the resilient consumer narrative leans on three ingredients that don\u2019t compound: a wealth effect concentrated in the top decile, a one-time refund bump, and a conflict-driven price pass-through.</p>

  <figure>
    <img src="{img(IMG5)}" alt="Figure 5">
    <figcaption>Figure 5: Retail sales vs Industrial Production (YoY)</figcaption>
  </figure>

  <h2 class="section">What would change our mind</h2>

  <p>The thesis breaks on two specific conditions.</p>

  <p>First, if April and May retail sales show food services and drinking places printing above 0.5% m/m in both months, and if clothing and sporting goods each clear +0.3%, the services side is re-accelerating and the goods-services split closes on its own. The CCI softening call would be wrong.</p>

  <p>Second, if industrial production prints positive m/m in either April or May and quits rate holds above 2.0% through the June release, the demand-supply gap is closing rather than widening, and the resilient consumer read earns its label.</p>

  <p>Absent both of those, this print stays in the data-artifact column.</p>

  <h2 class="section">The so-what</h2>

  <p>This favors staying neutral on cyclicals, because the labor-to-consumer lag means CCI prints the turn last. MRI regime hasn\u2019t softened on the strength of one nominal retail print. A 1.7% print with gasoline at 15.45% and food services at 0.14% is a composition story. That\u2019s what the framework is for.</p>

  <div class="footer">Lighthouse Macro | LighthouseMacro.com | @LHMacro</div>
</div>
</body>
</html>
'''

HTML_PATH.write_text(html)
size_mb = HTML_PATH.stat().st_size / (1024 * 1024)
print(f'Wrote {HTML_PATH} — {size_mb:.2f} MB')


## How to tweak — one-screen cheat sheet

**Annotation pill position** (axis coords, 0 = left/bottom, 1 = right/top):
- Top-left: `X=0.02, Y=0.95, HA='left'`
- Top-right: `X=0.98, Y=0.95, HA='right'`
- Bottom-left: `X=0.02, Y=0.14, HA='left'`
- Bottom-right: `X=0.98, Y=0.14, HA='right'`
- Center: `X=0.50, Y=0.95, HA='center'`
- Nudge up: increase Y (0.14 -> 0.25). Nudge right: increase X (0.80 -> 0.95).

**Legend placement** (`LEGEND_LOC`): `'upper left'`, `'upper right'`, `'lower left'`, `'lower right'`, `'center left'`, `'center right'`, `'upper center'`, `'lower center'`, `'center'`, `'best'`.

**Palette hexes** (copy for inline styling):
- Ocean `#2389BB` (primary), Dusk `#FF6723` (secondary)
- Sky `#23BBFF`, Sea `#00BB89`, Venus `#FF2389`
- Doldrums `#898989`, Fog `#D1D1D1`
- Starboard `#238923` (bullish), Port `#892323` (bearish)

**Change window start** (Fig 4, Fig 5): set `F4_WINDOW_START = '2023-01-01'` or `None` for full history.

**Override y-limits**: set `F4_YLIM = (-2, 20)` or `F5_YLIM = None` for auto.

**Extend Fig 2 bar padding**: bump `F2_LEFT_PAD_DAYS` (default 30) for more LHS breathing room.

**Fig 3 x-axis squeeze**: tighten with `F3_XLIM_LEFT = -0.95` (less pad past the -0.91% bar); loosen with `F3_XLIM_RIGHT_PAD = 2.5`.

**Workflow**: tweak the constants at the top of the cell, run just that cell, eyeball `plt.show()`, then run the HTML rebuild cell to refresh `beam_preview.html`.

**Don't edit below the TWEAKABLE block** — styling logic lives there. Brand rules are in the SETUP cell and they're locked.